# Fase 3: Inferensi Statistik (Confidence & Credible Intervals)
**Member:** Kevin Christman Lumban Tobing — Inference Analyst

**Pertanyaan Riset (P1):** Berapa estimasi probabilitas sebuah Pull Request (PR) diterima (merged) di repositori Pandas, beserta rata-rata harian frekuensi *issue* baru, dan seberapa besar rentang ketidakpastian dari estimasi tersebut?

**Tujuan:** Mengukur rentang ketidakpastian (*margin of error*) dari estimasi parameter titik (MLE) menggunakan pendekatan *Confidence Interval* (Frequentist) dan *Credible Interval* (Bayesian), serta memberikan interpretasi statistik yang akurat dan tepat secara konseptual.

## AI Usage Disclosure
**Member:** Kevin Christman Lumban Tobing — Inference Analyst | **Tools used:** Gemini

| Task                                             | Tool   | Prompt summary                                                                       | Output modified?                                       |
| ------------------------------------------------ | ------ | ------------------------------------------------------------------------------------ | ------------------------------------------------------ |
| Memeriksa sintaks fungsi library `scipy.stats`   | Gemini | "Bagaimana cara mendapatkan batas persentil distribusi Beta menggunakan SciPy di Python?" | Ya — disesuaikan dengan parameter variabel lokal saya. |

**Written entirely without AI:** Seluruh penulisan logika komputasi inferensi, konstruksi parameter batas *Confidence/Credible Interval*, dan penulisan narasi interpretasi konseptual (*Frequentist* & *Bayesian*) dikerjakan 100% manual tanpa bantuan AI.

In [9]:
import sys
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

sys.path.append("..")
# Import fungsi dari layer milik Member C sesuai persyaratan
from src.inference import confidence_interval, ci_bernoulli, ci_poisson, credible_interval

# Muat data bersih yang sudah disiapkan oleh Member A
df = pd.read_csv("../data/clean/dataset.csv")
df['created_at'] = pd.to_datetime(df['created_at'])

### 1. Confidence Interval untuk Rata-Rata Frekuensi Issue Harian (Poisson)
Menghitung interval kepercayaan 95% untuk parameter $\lambda$ (rata-rata harian *issue* baru). Karena ukuran sampel hari ($n$) cukup besar, kita dapat menggunakan pendekatan Distribusi Normal Baku (Z-distribution) berdasar pada *Central Limit Theorem*.

In [13]:
# Ekstrak data frekuensi harian dari DataFrame
df['date_only'] = df['created_at'].dt.date
daily_counts = df.groupby('date_only').size().values

# Kalkulasi Confidence Interval 95%
poisson_ci = ci_poisson(daily_counts, confidence=0.95)
print(f"95% Confidence Interval rata-rata issue harian (Lambda): [{poisson_ci[0]:.4f}, {poisson_ci[1]:.4f}]")

95% Confidence Interval rata-rata issue harian (Lambda): [14.0614, 15.3504]


**Interpretasi Frequentist (Poisson):**
Berdasarkan kalkulasi inferensial di atas, kita **yakin 95% bahwa interval rentang tersebut mencakup nilai parameter rata-rata populasi yang sebenarnya** ($\lambda$) dari frekuensi kemunculan *issue* harian di repositori Pandas. Dalam kerangka *Frequentist*, parameter populasi bersifat tetap (bukan variabel acak), sehingga interval ini mengindikasikan bahwa jika kita melakukan pengambilan sampel ulang tanpa batas dengan metode yang sama, sekitar 95% dari interval yang terbentuk akan berhasil menangkap rata-rata harian yang sesungguhnya.

### 2. Confidence Interval untuk Probabilitas Pull Request Diterima (Bernoulli)
Menghitung interval kepercayaan 95% untuk parameter $p$ (probabilitas suksesnya sebuah PR di-*merge* oleh tim *maintainer* Pandas).

In [14]:
# Ekstrak data status PR dari DataFrame
pr_data = df[df['is_pull_request'] == True]['is_merged'].astype(int).values
k = np.sum(pr_data)
n = len(pr_data)

# Kalkulasi Confidence Interval 95%
bernoulli_ci = ci_bernoulli(k, n, confidence=0.95)
print(f"95% Confidence Interval probabilitas PR diterima (p): [{bernoulli_ci[0]:.4f}, {bernoulli_ci[1]:.4f}]")

95% Confidence Interval probabilitas PR diterima (p): [0.6214, 0.6665]


**Interpretasi Frequentist (Bernoulli):**
Kita **yakin 95% bahwa rentang interval yang dihasilkan mencakup probabilitas populasi yang sebenarnya** ($p$) terkait tingkat penerimaan *Pull Request*. Penting untuk dicatat bahwa interval ini tidak merepresentasikan probabilitas sebesar 95% bahwa parameter berada di dalamnya, melainkan merepresentasikan tingkat keandalan metode pengukuran statistik kita dalam menangkap nilai parameter yang tidak diketahui.

### 3. Credible Interval untuk Probabilitas Pull Request Diterima (Pendekatan Bayesian)
Sebagai bentuk evaluasi komparatif dari metrik *Frequentist*, kita mengukur batas ketidakpastian menggunakan pendekatan Bayesian. *Credible Interval* dihitung secara langsung dari luasan area fungsi Distribusi Posterior Beta($\alpha, \beta$). Sesuai dengan spesifikasi parameter dari fase estimasi (Buku Tsun, 2020), kita menetapkan $\alpha = k+1$ dan $\beta = m+1$.

In [15]:
# Kalkulasi jumlah sukses (k) dan gagal (m)
m = n - k

# Konjugasi parameter posterior
alpha_post = k + 1
beta_post = m + 1

# Kalkulasi Credible Interval 95%
cred_int = credible_interval(alpha_post, beta_post, confidence=0.95)
print(f"95% Credible Interval (Bayesian) peluang PR diterima (p): [{cred_int[0]:.4f}, {cred_int[1]:.4f}]")

95% Credible Interval (Bayesian) peluang PR diterima (p): [0.6211, 0.6662]


**Interpretasi Bayesian:**
Perbedaan fundamental terjadi pada penarikan kesimpulan Bayesian. Karena probabilitas diperlakukan sebagai representasi tingkat keyakinan (*degree of belief*), kita dapat membuat pernyataan probabilistik secara langsung. Mengacu pada distribusi posterior Beta di atas, kita dapat menyimpulkan bahwa **terdapat probabilitas sebesar 95% bahwa nilai peluang sesungguhnya ($p$) dari sebuah Pull Request untuk di-merge terletak tepat di dalam interval ini**.

### Kesimpulan Handoff
Tahap Inferensi Statistik (*Confidence & Credible Intervals*) telah diselesaikan dengan sukses. Evaluasi rentang ketidakpastian (*margin of error*) untuk parameter rata-rata *issue* (Poisson) dan rasio sukses *Pull Request* (Bernoulli) telah dikalkulasi dan diinterpretasikan secara rigid. Distribusi data yang telah teruji tingkat keandalannya ini telah sah dan siap diserahkan kepada **Safani (Member D)** untuk dilanjutkan ke fase Pengujian Hipotesis (Modul 04) menggunakan Z-Test.